# CIFAR-10 Adversarial Attack and Defense Evaluation

This notebook is the reproducible evaluation entry point for the
project. It operates on raw CIFAR-10 images in `[0, 1]`; input
normalization remains inside the model.

The notebook supports two modes:

- `smoke`: small deterministic subsets for structural validation.
- `final`: project-scale evaluation and report generation.

Main responsibilities:

1. Restore checkpoints from Google Drive.
2. Load the clean, PGD-AT, and Gaussian-fine-tuned classifiers.
3. Load attack and defense implementations.
4. Run deterministic smoke checks before expensive evaluation.
5. Produce the final metrics table and project figures.

## 1. Runtime and repository setup

This section mounts Google Drive and clones the repository when
necessary. If the repository already exists and is clean, it is
updated with a fast-forward-only pull.

In [ ]:
from pathlib import Path
import os
import subprocess

from google.colab import drive

REPOSITORY_URL = (
    "https://github.com/MostafaOmidi17/"
    "AI-Project-Adversarial-Attack-Defense.git"
)

PROJECT_ROOT = Path(
    "/content/AI-Project-Adversarial-Attack-Defense"
)

drive.mount("/content/drive")

if not (PROJECT_ROOT / ".git").is_dir():
    subprocess.run(
        [
            "git",
            "clone",
            REPOSITORY_URL,
            str(PROJECT_ROOT),
        ],
        check=True,
    )
else:
    repository_status = subprocess.run(
        [
            "git",
            "-C",
            str(PROJECT_ROOT),
            "status",
            "--porcelain",
        ],
        check=True,
        text=True,
        capture_output=True,
    ).stdout.strip()

    if repository_status:
        print(
            "Repository contains local changes; "
            "automatic pull was skipped."
        )
    else:
        subprocess.run(
            [
                "git",
                "-C",
                str(PROJECT_ROOT),
                "pull",
                "--ff-only",
            ],
            check=True,
        )

os.chdir(PROJECT_ROOT)

current_commit = subprocess.run(
    [
        "git",
        "rev-parse",
        "--short",
        "HEAD",
    ],
    check=True,
    text=True,
    capture_output=True,
).stdout.strip()

print("Project root:", PROJECT_ROOT)
print("Git commit:", current_commit)

## 2. Dependencies

Requirements are installed inside the current Colab runtime.
Diffusion-related dependencies are listed explicitly because the
purification model is loaded from Hugging Face.

In [ ]:
import subprocess
import sys

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "-r",
        str(PROJECT_ROOT / "requirements.txt"),
    ],
    check=True,
)

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "diffusers",
        "huggingface_hub",
        "safetensors",
        "accelerate",
    ],
    check=True,
)

print("Dependencies installed.")

## 3. Reproducibility, paths, and evaluation mode

`smoke` mode must pass before `final` mode is enabled.

In [ ]:
import hashlib
import json
import random
import shutil
import time

import numpy as np
import pandas as pd
import torch
import torchvision

from torch.utils.data import DataLoader, Subset
from torchvision import transforms

SEED = 42
RUN_MODE = "smoke"

if RUN_MODE not in {"smoke", "final"}:
    raise ValueError(
        "RUN_MODE must be 'smoke' or 'final'."
    )

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device(
    "cuda:0"
    if torch.cuda.is_available()
    else "cpu"
)

CHECKPOINT_DIR = PROJECT_ROOT / "checkpoints"
DATA_DIR = PROJECT_ROOT / "data"
RESULTS_DIR = PROJECT_ROOT / "results"
FIGURES_DIR = PROJECT_ROOT / "figures"

DRIVE_PROJECT_DIR = Path(
    "/content/drive/MyDrive/"
    "AI-Project-Adversarial-Attack-Defense"
)

DRIVE_CHECKPOINT_DIR = (
    DRIVE_PROJECT_DIR / "checkpoints"
)

DRIVE_RESULTS_DIR = (
    DRIVE_PROJECT_DIR / "results"
)

for directory in (
    CHECKPOINT_DIR,
    DATA_DIR,
    RESULTS_DIR,
    FIGURES_DIR,
    DRIVE_CHECKPOINT_DIR,
    DRIVE_RESULTS_DIR,
):
    directory.mkdir(
        parents=True,
        exist_ok=True,
    )

MODE_CONFIG = {
    "smoke": {
        "general_samples": 64,
        "expensive_attack_samples": 4,
        "diffusion_samples": 2,
    },
    "final": {
        "general_samples": 10000,
        "expensive_attack_samples": 1000,
        "diffusion_samples": 200,
    },
}

active_config = MODE_CONFIG[RUN_MODE]

print("Run mode:", RUN_MODE)
print("Device:", device)
print("Configuration:", active_config)

## 4. Restore and verify checkpoints

Checkpoints are not stored in Git. They are restored from Google
Drive and verified using known SHA-256 hashes.

In [ ]:
EXPECTED_CHECKPOINT_HASHES = {
    "resnet20_clean_best.pt":
        "8874dc56b8fd452aba4cba76774b3e3cab6e16f3740ab6437c3e924743f5c9b0",

    "resnet20_pgd_at_eps8_best.pt":
        "e63b5f7f483453203bb0224d6967b9b88ae28f68e72e1d8a99bfc66738e94210",

    "resnet20_pgd_at_eps8_last.pt":
        "28e0ced68d513ee320d95d007fd5953d2b98a62b626f7988f0ec4d157fd78872",

    "resnet20_rand_smooth_sigma0p25_best.pt":
        "c5f7c857c1c43d4df5b3376a404c5f7faa4e554ed6d6867291e9cda8b2dbc238",

    "resnet20_rand_smooth_sigma0p25_last.pt":
        "92506d2cb180574e4eab5b681e8df41c5309e1fc60506d8590313b0c717fc324",
}


def calculate_sha256(path: Path) -> str:
    with path.open("rb") as checkpoint_file:
        return hashlib.file_digest(
            checkpoint_file,
            "sha256",
        ).hexdigest()


for checkpoint_name, expected_hash in (
    EXPECTED_CHECKPOINT_HASHES.items()
):
    drive_path = (
        DRIVE_CHECKPOINT_DIR / checkpoint_name
    )

    local_path = (
        CHECKPOINT_DIR / checkpoint_name
    )

    if not drive_path.is_file():
        raise FileNotFoundError(
            f"Drive checkpoint is missing: {drive_path}"
        )

    if (
        not local_path.is_file()
        or calculate_sha256(local_path)
        != expected_hash
    ):
        shutil.copy2(
            drive_path,
            local_path,
        )

    actual_hash = calculate_sha256(
        local_path
    )

    if actual_hash != expected_hash:
        raise RuntimeError(
            f"Checkpoint hash mismatch: "
            f"{checkpoint_name}"
        )

    print(
        f"VERIFIED: {checkpoint_name} "
        f"({local_path.stat().st_size:,} bytes)"
    )

print(
    "\nALL CHECKPOINTS RESTORED AND VERIFIED."
)

## 5. CIFAR-10 test data

The external transform is only `ToTensor()`. Normalization is
performed by the model. Deterministic nested subsets are used so
that expensive methods evaluate the same initial test examples.

In [ ]:
test_dataset = torchvision.datasets.CIFAR10(
    root=str(DATA_DIR),
    train=False,
    download=True,
    transform=transforms.ToTensor(),
)

subset_generator = (
    torch.Generator().manual_seed(SEED)
)

fixed_permutation = torch.randperm(
    len(test_dataset),
    generator=subset_generator,
).tolist()

general_sample_count = min(
    active_config["general_samples"],
    len(test_dataset),
)

expensive_sample_count = min(
    active_config[
        "expensive_attack_samples"
    ],
    general_sample_count,
)

diffusion_sample_count = min(
    active_config["diffusion_samples"],
    general_sample_count,
)

general_indices = fixed_permutation[
    :general_sample_count
]

expensive_indices = fixed_permutation[
    :expensive_sample_count
]

diffusion_indices = fixed_permutation[
    :diffusion_sample_count
]

general_dataset = Subset(
    test_dataset,
    general_indices,
)

expensive_dataset = Subset(
    test_dataset,
    expensive_indices,
)

diffusion_dataset = Subset(
    test_dataset,
    diffusion_indices,
)

general_loader = DataLoader(
    general_dataset,
    batch_size=128,
    shuffle=False,
    num_workers=2,
    pin_memory=torch.cuda.is_available(),
)

expensive_loader = DataLoader(
    expensive_dataset,
    batch_size=4,
    shuffle=False,
    num_workers=2,
    pin_memory=torch.cuda.is_available(),
)

diffusion_loader = DataLoader(
    diffusion_dataset,
    batch_size=2,
    shuffle=False,
    num_workers=2,
    pin_memory=torch.cuda.is_available(),
)

sample_images, sample_labels = next(
    iter(general_loader)
)

assert sample_images.shape[1:] == (
    3,
    32,
    32,
)

assert sample_images.min().item() >= 0.0
assert sample_images.max().item() <= 1.0

print("Full test size:", len(test_dataset))
print("General subset:", len(general_dataset))
print(
    "Expensive-attack subset:",
    len(expensive_dataset),
)
print(
    "Diffusion subset:",
    len(diffusion_dataset),
)
print("CIFAR-10 DATA SETUP PASSED.")

## 6. Load project modules and classifier checkpoints

In [ ]:
from src.models import (
    build_normalized_resnet20,
)

from src.attacks.fgsm import fgsm_attack
from src.attacks.pgd import pgd_attack
from src.attacks.deepfool import deepfool_attack
from src.attacks.cw_l2 import cw_l2_attack

from src.defenses.randomized_smoothing import (
    predict_smoothed,
)

from src.defenses.diffusion_purification import (
    DEFAULT_DIFFUSION_MODEL_ID,
    load_diffusion_components,
    purify_images,
)

clean_model = build_normalized_resnet20(
    checkpoint_path=(
        CHECKPOINT_DIR
        / "resnet20_clean_best.pt"
    ),
    device=device,
    eval_mode=True,
)

pgd_at_model = build_normalized_resnet20(
    checkpoint_path=(
        CHECKPOINT_DIR
        / "resnet20_pgd_at_eps8_best.pt"
    ),
    device=device,
    eval_mode=True,
)

smoothing_model = build_normalized_resnet20(
    checkpoint_path=(
        CHECKPOINT_DIR
        / "resnet20_rand_smooth_sigma0p25_best.pt"
    ),
    device=device,
    eval_mode=True,
)

classifier_models = {
    "none": clean_model,
    "pgd_at": pgd_at_model,
    "rand_smooth": smoothing_model,
}

for defense_id, model in (
    classifier_models.items()
):
    assert (
        next(model.parameters()).device
        == device
    )

    print(
        f"Loaded {defense_id} on "
        f"{next(model.parameters()).device}"
    )

print("CLASSIFIER CHECKPOINT LOADING PASSED.")

## 7. Structural smoke test

This test performs very small attack calls. It does not write
final metrics and does not load the diffusion model yet.

In [ ]:
smoke_images = sample_images[:2].to(device)
smoke_labels = sample_labels[:2].to(device)

for defense_id, model in (
    classifier_models.items()
):
    with torch.no_grad():
        logits = model(smoke_images)

    assert logits.shape == (2, 10)
    assert torch.isfinite(logits).all()

    print(
        f"{defense_id} forward PASSED: "
        f"{tuple(logits.shape)}"
    )

smoke_fgsm = fgsm_attack(
    model=clean_model,
    images=smoke_images,
    labels=smoke_labels,
    epsilon=8 / 255,
)

smoke_pgd = pgd_attack(
    model=clean_model,
    images=smoke_images,
    labels=smoke_labels,
    epsilon=8 / 255,
    alpha=2 / 255,
    steps=2,
    random_start=True,
    restarts=1,
)

smoke_deepfool = deepfool_attack(
    model=clean_model,
    images=smoke_images,
    labels=smoke_labels,
    max_steps=1,
    overshoot=0.02,
    num_classes=10,
)

smoke_cw = cw_l2_attack(
    model=clean_model,
    images=smoke_images,
    labels=smoke_labels,
    c=1.0,
    kappa=0.0,
    learning_rate=0.01,
    steps=2,
)

for attack_id, adversarial_images in {
    "fgsm": smoke_fgsm,
    "pgd": smoke_pgd,
    "deepfool": smoke_deepfool,
    "cw_l2": smoke_cw,
}.items():
    assert (
        adversarial_images.shape
        == smoke_images.shape
    )

    assert (
        adversarial_images.min().item()
        >= 0.0
    )

    assert (
        adversarial_images.max().item()
        <= 1.0
    )

    print(
        f"{attack_id} structural call PASSED."
    )

smoothing_generator = torch.Generator(
    device=device,
).manual_seed(SEED)

(
    smoke_smoothed_predictions,
    smoke_vote_counts,
) = predict_smoothed(
    model=smoothing_model,
    images=smoke_images,
    sigma=0.25,
    num_samples=7,
    noise_batch_size=3,
    num_classes=10,
    clip_noise=True,
    generator=smoothing_generator,
)

assert (
    smoke_smoothed_predictions.shape
    == (2,)
)

assert smoke_vote_counts.shape == (
    2,
    10,
)

assert torch.all(
    smoke_vote_counts.sum(dim=1) == 7
)

print(
    "Randomized smoothing structural call PASSED."
)

print(
    "\nEVALUATION NOTEBOOK PHASE-1 "
    "SMOKE TEST PASSED."
)

## Next phase

After this setup passes in a fresh Colab runtime, the notebook
will be extended with:

- contract-compliant metric aggregation;
- clean, FGSM, PGD, DeepFool, and C&W evaluation;
- randomized-smoothing defended predictions;
- diffusion purification on the fixed subset;
- CSV/config persistence and final visualization generation.